In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Load the dataset
df = pd.read_csv('/content/merged_time_series_data.csv')

# Display basic information and first few rows to locate price and quantity columns
print("Columns in dataset:", df.columns.tolist())
display(df.head())


Columns in dataset: ['Unnamed: 0', 'overall_job_postings', 'ppi_data_processing_services', 'software_job_postings', 'merged_job_postings_di', 'nvidia_quarterly_revenue (millions)']


,Unnamed: 0,overall_job_postings,ppi_data_processing_services,software_job_postings,merged_job_postings_di,nvidia_quarterly_revenue (millions)
0,6/1/21,2551,112,2636,133,"$5,661.00"
1,7/1/21,4258,112,4518,137,"$6,507.00"
2,8/1/21,4313,112,4905,139,"$6,507.00"
3,9/1/21,4294,112,5209,143,"$6,507.00"
4,10/1/21,4556,112,5776,147,"$7,103.00"


Once we run the above cell, we can define our log-log model. A standard log-log regression formulation for Cross Price Elasticity with distributed lags looks like:

$$\ln(Q_{A, t}) = \alpha + \sum_{i=0}^{L} \beta_i \ln(P_{B, t-i}) + \gamma \ln(P_{A, t}) + \epsilon_t$$

Where:
* $Q_{A, t}$ is the quantity demanded of Product A at time $t$.
* $P_{B, t-i}$ is the price of Product B at lag $i$.
* $\beta_i$ represents the cross-price elasticity at lag $i$. The sum of these coefficients represents the cumulative/long-run cross-price elasticity.
* $P_{A, t}$ is own price to control for own-price effects.

In [2]:
# Placeholder code assuming column names. Please adjust 'quantity_A', 'price_A', and 'price_B' to match your actual column names from the preview above.
# We will use 3 lags as an example for the distributed lag structure.

def calculate_cpe_with_lags(df, qty_col='quantity_A', own_price_col='price_A', cross_price_col='price_B', lags=3):
    # Avoid modification warning
    data = df[[qty_col, own_price_col, cross_price_col]].copy().dropna()

    # Take logs of the variables
    data['log_qty'] = np.log(data[qty_col])
    data['log_own_price'] = np.log(data[own_price_col])
    data['log_cross_price'] = np.log(data[cross_price_col])

    # Create distributed lags for log of cross price
    lag_cols = []
    for i in range(lags + 1):
        col_name = f'log_cross_price_lag_{i}'
        data[col_name] = data['log_cross_price'].shift(i)
        lag_cols.append(col_name)

    # Drop rows with NaN values resulting from shifting
    data = data.dropna()

    # Define independent variables (X) and dependent variable (y)
    X = data[['log_own_price'] + lag_cols]
    X = sm.add_constant(X)
    y = data['log_qty']

    # Fit the OLS regression model
    model = sm.OLS(y, X).fit()

    print(model.summary())

    # Cumulative cross-price elasticity calculation
    cumulative_cpe = sum(model.params[lag_cols])
    print(f"\nCumulative Cross Price Elasticity (Sum of lags 0 to {lags}): {cumulative_cpe:.4f}")

    return model

# Note: Run this function by substituting actual column names once you verify the dataframe columns.
# example: model = calculate_cpe_with_lags(df, qty_col='YOUR_QTY_COL', own_price_col='YOUR_OWN_PRICE_COL', cross_price_col='YOUR_CROSS_PRICE_COL')


In [3]:
# Clean the NVIDIA Quarterly Revenue column to make it numeric
df['nvidia_quarterly_revenue_clean'] = df['nvidia_quarterly_revenue (millions)'].astype(str).str.replace('$', '').str.replace(',', '').astype(float)

# Let's run the model:
# Quantity (Qty): 'software_job_postings'
# Own price proxy: 'ppi_data_processing_services'
# Cross price/revenue proxy: 'nvidia_quarterly_revenue_clean'
# We will use 3 lags for the distributed lag structure.

print("Fitting the double-log model with distributed lags...")
model = calculate_cpe_with_lags(
    df=df,
    qty_col='software_job_postings',
    own_price_col='ppi_data_processing_services',
    cross_price_col='nvidia_quarterly_revenue_clean',
    lags=3
)

Fitting the double-log model with distributed lags...
                            OLS Regression Results                            
Dep. Variable:                log_qty   R-squared:                       0.744
Model:                            OLS   Adj. R-squared:                  0.719
Method:                 Least Squares   F-statistic:                     29.68
Date:                Mon, 12 Oct 2026   Prob (F-statistic):           5.40e-14
Time:                        21:52:20   Log-Likelihood:                 1.9395
No. Observations:                  57   AIC:                             8.121
Df Residuals:                      51   BIC:                             20.38
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------

In [4]:
# @title 📊 Cross Price Elasticity (XED) Explorer {display-mode: "form"}

import pandas as pd
import numpy as np
import json
import statsmodels.api as sm
from google.colab import output
from IPython.display import HTML

# Ensure clean dataset
df = pd.read_csv('/content/merged_time_series_data.csv')
df['nvidia_quarterly_revenue_clean'] = df['nvidia_quarterly_revenue (millions)'].astype(str).str.replace('$', '').str.replace(',', '').astype(float)
df = df.dropna()

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

def compute_all_elasticities():
    # Target quantity variables
    qty_cols = ['overall_job_postings', 'software_job_postings', 'merged_job_postings_di']
    # Price variables
    price_cols = ['ppi_data_processing_services', 'nvidia_quarterly_revenue_clean']

    results = {}

    for qty in qty_cols:
        results[qty] = {}
        for price in price_cols:
            # Fit distributed lags (0 to 3 lags)
            temp_df = df[[qty, price]].copy().dropna()
            # Prevent zeros causing log issues
            temp_df = temp_df[(temp_df[qty] > 0) & (temp_df[price] > 0)]

            temp_df['log_qty'] = np.log(temp_df[qty])
            temp_df['log_price'] = np.log(temp_df[price])

            lag_cols = []
            for l in range(4):
                col_name = f'log_price_lag_{l}'
                temp_df[col_name] = temp_df['log_price'].shift(l)
                lag_cols.append(col_name)

            temp_df = temp_df.dropna()

            X = temp_df[lag_cols]
            X = sm.add_constant(X)
            y = temp_df['log_qty']

            model = sm.OLS(y, X).fit()

            lags_coef = []
            p_values = []
            for col in lag_cols:
                lags_coef.append(float(model.params[col]))
                p_values.append(float(model.pvalues[col]))

            cumulative_cpe = float(sum(model.params[lag_cols]))

            results[qty][price] = {
                'lags_coef': lags_coef,
                'p_values': p_values,
                'cumulative': cumulative_cpe,
                'r2': float(model.rsquared),
                'dw': float(sm.stats.stattools.durbin_watson(model.resid))
            }
    return results

# Prepare JSON data for JS consumption
raw_data = df.to_dict(orient='records')
elasticity_results = compute_all_elasticities()

html_content = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>XED Dashboard</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        body {
            background-color: #f4f6f8;
            font-family: ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
        }
        .canvas-wrapper {
            position: relative;
            flex-grow: 1;
            min-height: 0;
        }
    </style>
</head>
<body class="p-6">

    <!-- Header & Dashboard Controls -->
    <div class="max-w-7xl mx-auto space-y-6">
        <div class="flex flex-col md:flex-row md:items-center justify-between bg-white p-6 rounded-xl shadow-sm border border-gray-100">
            <div>
                <h1 class="text-2xl font-bold text-gray-800">Cross Price Elasticity (XED) Dashboard</h1>
                <p class="text-sm text-gray-500">Explore short-run and long-run elasticities with distributed lags (0 to 3 quarters).</p>
            </div>
            <div class="mt-4 md:mt-0 flex gap-4">
                <div>
                    <label class="block text-xs font-semibold text-gray-600 uppercase mb-1">Job Posting Class (Quantity)</label>
                    <select id="qty-select" class="bg-gray-50 border border-gray-200 rounded-lg p-2 text-sm text-gray-700 outline-none focus:ring-2 focus:ring-blue-500">
                        <option value="software_job_postings" selected>Software Job Postings</option>
                        <option value="overall_job_postings">Overall Job Postings</option>
                        <option value="merged_job_postings_di">Merged Job Postings DI</option>
                    </select>
                </div>
                <div>
                    <label class="block text-xs font-semibold text-gray-600 uppercase mb-1">Price/Revenue Variable</label>
                    <select id="price-select" class="bg-gray-50 border border-gray-200 rounded-lg p-2 text-sm text-gray-700 outline-none focus:ring-2 focus:ring-blue-500">
                        <option value="nvidia_quarterly_revenue_clean" selected>Nvidia Quarterly Revenue</option>
                        <option value="ppi_data_processing_services">PPI Data Processing Services</option>
                    </select>
                </div>
            </div>
        </div>

        <!-- Metric Summary Cards -->
        <div class="grid grid-cols-1 md:grid-cols-4 gap-6">
            <div class="bg-white p-6 rounded-xl shadow-sm border border-gray-100">
                <h3 class="text-xs font-semibold text-gray-500 uppercase tracking-wider">Cumulative (Long-Run) XED</h3>
                <p id="card-cumulative" class="text-3xl font-bold text-blue-600 mt-2">0.00</p>
                <p class="text-xs text-gray-400 mt-1">Sum of all lag coefficients</p>
            </div>
            <div class="bg-white p-6 rounded-xl shadow-sm border border-gray-100">
                <h3 class="text-xs font-semibold text-gray-500 uppercase tracking-wider">Immediate Impact (Lag 0)</h3>
                <p id="card-lag0" class="text-3xl font-bold text-gray-800 mt-2">0.00</p>
                <p class="text-xs text-gray-400 mt-1">Short-run elasticity</p>
            </div>
            <div class="bg-white p-6 rounded-xl shadow-sm border border-gray-100">
                <h3 class="text-xs font-semibold text-gray-500 uppercase tracking-wider">Model R-squared</h3>
                <p id="card-r2" class="text-3xl font-bold text-green-600 mt-2">0.00</p>
                <p class="text-xs text-gray-400 mt-1">Explained variance of quantity</p>
            </div>
            <div class="bg-white p-6 rounded-xl shadow-sm border border-gray-100">
                <h3 class="text-xs font-semibold text-gray-500 uppercase tracking-wider">Durbin-Watson Stat</h3>
                <p id="card-dw" class="text-3xl font-bold text-orange-600 mt-2">0.00</p>
                <p class="text-xs text-gray-400 mt-1">Value < 1.5 indicates autocorrelation</p>
            </div>
        </div>

        <!-- Mid Charts Section -->
        <div class="grid grid-cols-1 lg:grid-cols-3 gap-6">
            <!-- Lag Distribution Bar Chart -->
            <div class="bg-white p-6 rounded-xl shadow-sm border border-gray-100 lg:col-span-2 flex flex-col h-[400px]">
                <h3 class="text-md font-semibold text-gray-800 mb-4">Elasticity Coefficients by Lag (Double-Log Structure)</h3>
                <div class="canvas-wrapper">
                    <canvas id="lagChart"></canvas>
                </div>
            </div>

            <!-- Structured Interpretation Panel -->
            <div class="bg-white p-6 rounded-xl shadow-sm border border-gray-100 flex flex-col justify-between">
                <div>
                    <h3 class="text-md font-semibold text-gray-800 mb-3">Economic Interpretation</h3>
                    <div class="space-y-4 text-sm text-gray-600">
                        <div>
                            <span class="font-medium text-gray-800">Relationship Type:</span>
                            <p id="interpret-relation" class="mt-1"></p>
                        </div>
                        <div>
                            <span class="font-medium text-gray-800">Elasticity Strength:</span>
                            <p id="interpret-strength" class="mt-1"></p>
                        </div>
                        <div>
                            <span class="font-medium text-gray-800">Time-Lag Dynamic:</span>
                            <p id="interpret-lag" class="mt-1"></p>
                        </div>
                    </div>
                </div>
                <div class="mt-4 pt-4 border-t border-gray-100 text-xs text-gray-400">
                    Interpretation dynamically computed from standard OLS double-log coefficients.
                </div>
            </div>
        </div>

        <!-- Bottom Trend Visualization -->
        <div class="bg-white p-6 rounded-xl shadow-sm border border-gray-100 flex flex-col h-[350px]">
            <h3 class="text-md font-semibold text-gray-800 mb-4">Normalized Time Series Comparison</h3>
            <div class="canvas-wrapper">
                <canvas id="trendChart"></canvas>
            </div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const dataset = DATA_PLACEHOLDER;
        const elasticities = ELASTICITY_PLACEHOLDER;

        const qtySelect = document.getElementById('qty-select');
        const priceSelect = document.getElementById('price-select');

        let lagChartInstance = null;
        let trendChartInstance = null;

        function updateDashboard() {
            const qty = qtySelect.value;
            const price = priceSelect.value;
            const metrics = elasticities[qty][price];

            // Update metric summary cards
            document.getElementById('card-cumulative').innerText = metrics.cumulative.toFixed(4);
            document.getElementById('card-lag0').innerText = metrics.lags_coef[0].toFixed(4);
            document.getElementById('card-r2').innerText = metrics.r2.toFixed(3);
            document.getElementById('card-dw').innerText = metrics.dw.toFixed(3);

            // Write Interpretations
            const relText = metrics.cumulative > 0
                ? "These goods exhibit a positive long-run cumulative XED, suggesting they behave as Substitutes. An increase in the price/revenue of the benchmark variable pushes demand up for this category overall."
                : "These goods exhibit a negative long-run cumulative XED, behaving as Complementary services or products. Increases in the price/revenue benchmark dampens demand.";
            document.getElementById('interpret-relation').innerText = relText;

            const absCum = Math.abs(metrics.cumulative);
            const strengthText = absCum > 1
                ? `Highly Elastic (${absCum.toFixed(2)}): Demand is extremely sensitive to changes in the independent metric.`
                : `Inelastic (${absCum.toFixed(2)}): Demand responds relatively weakly to changes in the independent metric.`;
            document.getElementById('interpret-strength').innerText = strengthText;

            const maxLagVal = Math.max(...metrics.lags_coef.map(Math.abs));
            const peakLag = metrics.lags_coef.findIndex(v => Math.abs(v) === maxLagVal);
            document.getElementById('interpret-lag').innerText = `The peak magnitude of impact occurs at Lag ${peakLag} (coefficient: ${metrics.lags_coef[peakLag].toFixed(4)}), implying that the effect takes roughly ${peakLag} quarters to completely manifest in demand dynamics.`;

            // Chart 1: Lags Coefficient Chart
            if (lagChartInstance) lagChartInstance.destroy();
            const ctx1 = document.getElementById('lagChart').getContext('2d');
            lagChartInstance = new Chart(ctx1, {
                type: 'bar',
                data: {
                    labels: ['Lag 0 (Short Run)', 'Lag 1', 'Lag 2', 'Lag 3'],
                    datasets: [{
                        label: 'Elasticity Coefficient',
                        data: metrics.lags_coef,
                        backgroundColor: metrics.lags_coef.map(v => v >= 0 ? 'rgba(59, 130, 246, 0.7)' : 'rgba(239, 68, 68, 0.7)'),
                        borderColor: metrics.lags_coef.map(v => v >= 0 ? 'rgb(59, 130, 246)' : 'rgb(239, 68, 68)'),
                        borderWidth: 1.5
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: { display: false }
                    },
                    scales: {
                        y: {
                            title: { display: true, text: 'Elasticity (% change in Q per 1% change in P)' },
                            grid: { color: '#e5e7eb' }
                        },
                        x: { grid: { display: false } }
                    }
                }
            });

            // Chart 2: Time Series Normalized Trend comparison
            // Min-max normalization for clean visual comparison
            const dates = dataset.map(row => row['Unnamed: 0'] || '');
            const rawQty = dataset.map(row => row[qty] || 0);
            const rawPrice = dataset.map(row => row[price] || 0);

            const minQty = Math.min(...rawQty), maxQty = Math.max(...rawQty);
            const minPrice = Math.min(...rawPrice), maxPrice = Math.max(...rawPrice);

            const normQty = rawQty.map(v => (v - minQty) / (maxQty - minQty || 1));
            const normPrice = rawPrice.map(v => (v - minPrice) / (maxPrice - minPrice || 1));

            if (trendChartInstance) trendChartInstance.destroy();
            const ctx2 = document.getElementById('trendChart').getContext('2d');
            trendChartInstance = new Chart(ctx2, {
                type: 'line',
                data: {
                    labels: dates,
                    datasets: [
                        {
                            label: `Normalized Job Postings (Quantity: ${qty})`,
                            data: normQty,
                            borderColor: '#3b82f6',
                            backgroundColor: 'rgba(59, 130, 246, 0.05)',
                            tension: 0.2,
                            fill: true
                        },
                        {
                            label: `Normalized Price / Metric (Price: ${price})`,
                            data: normPrice,
                            borderColor: '#f97316',
                            backgroundColor: 'transparent',
                            tension: 0.2
                        }
                    ]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: {
                        legend: { position: 'top' }
                    },
                    scales: {
                        y: { max: 1.1, min: -0.1, grid: { color: '#e5e7eb' }, title: { display: true, text: 'Normalized Value (0 to 1)' } },
                        x: { grid: { display: false } }
                    }
                }
            });
        }

        qtySelect.addEventListener('change', updateDashboard);
        priceSelect.addEventListener('change', updateDashboard);

        // Init dashboard
        updateDashboard();
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json.dumps(raw_data)).replace('ELASTICITY_PLACEHOLDER', json.dumps(elasticity_results))

HTML(html_content)


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Ensure clean dataset
df = pd.read_csv('/content/merged_time_series_data.csv')
df['nvidia_quarterly_revenue_clean'] = df['nvidia_quarterly_revenue (millions)'].astype(str).str.replace('$', '').str.replace(',', '').astype(float)
df = df.dropna()

qty_cols = ['overall_job_postings', 'software_job_postings', 'merged_job_postings_di']
price_cols = ['ppi_data_processing_services', 'nvidia_quarterly_revenue_clean']

summary_records = []

for qty in qty_cols:
    for price in price_cols:
        temp_df = df[[qty, price]].copy().dropna()
        temp_df = temp_df[(temp_df[qty] > 0) & (temp_df[price] > 0)]

        temp_df['log_qty'] = np.log(temp_df[qty])
        temp_df['log_price'] = np.log(temp_df[price])

        lag_cols = []
        for l in range(4):
            col_name = f'log_price_lag_{l}'
            temp_df[col_name] = temp_df['log_price'].shift(l)
            lag_cols.append(col_name)

        temp_df = temp_df.dropna()

        X = temp_df[lag_cols]
        X = sm.add_constant(X)
        y = temp_df['log_qty']

        model = sm.OLS(y, X).fit()

        record = {
            'quantity_variable': qty,
            'price_variable': price,
            'r_squared': model.rsquared,
            'adj_r_squared': model.rsquared_adj,
            'durbin_watson': sm.stats.stattools.durbin_watson(model.resid),
            'intercept_coef': model.params['const'],
            'lag_0_coef': model.params['log_price_lag_0'],
            'lag_1_coef': model.params['log_price_lag_1'],
            'lag_2_coef': model.params['log_price_lag_2'],
            'lag_3_coef': model.params['log_price_lag_3'],
            'cumulative_xed': sum(model.params[lag_cols]),
            'lag_0_pvalue': model.pvalues['log_price_lag_0'],
            'lag_1_pvalue': model.pvalues['log_price_lag_1'],
            'lag_2_pvalue': model.pvalues['log_price_lag_2'],
            'lag_3_pvalue': model.pvalues['log_price_lag_3']
        }
        summary_records.append(record)

# Create summary DataFrame and export to CSV
summary_df = pd.DataFrame(summary_records)
export_path = '/content/elasticity_summary_statistics.csv'
summary_df.to_csv(export_path, index=False)

print(f"Summary statistics exported successfully to {export_path}")
display(summary_df)


Summary statistics exported successfully to /content/elasticity_summary_statistics.csv


,quantity_variable,price_variable,r_squared,adj_r_squared,durbin_watson,intercept_coef,lag_0_coef,lag_1_coef,lag_2_coef,lag_3_coef,cumulative_xed,lag_0_pvalue,lag_1_pvalue,lag_2_pvalue,lag_3_pvalue
0,overall_job_postings,ppi_data_processing_services,0.842123,0.829978,0.709993,36.290330,-2.206907,-1.219162,-1.352412,-1.127329,-5.905810,0.088558,0.440354,0.398878,0.401580
1,overall_job_postings,nvidia_quarterly_revenue_clean,0.869883,0.859874,0.798903,9.867034,-0.168866,-0.069136,0.006647,0.066261,-0.165094,0.024080,0.457669,0.943218,0.373460
2,software_job_postings,ppi_data_processing_services,0.653522,0.626870,0.086422,79.828446,-7.091654,-4.782723,-2.705039,-0.544415,-15.123831,0.203192,0.484046,0.695817,0.925173
3,software_job_postings,nvidia_quarterly_revenue_clean,0.711212,0.688997,0.186194,11.896993,-1.018994,-0.070879,-0.013445,0.714316,-0.389002,0.002198,0.860767,0.973592,0.030530
4,merged_job_postings_di,ppi_data_processing_services,0.872902,0.863125,0.167711,33.042779,-2.253242,-0.903658,-0.831355,-1.953001,-5.941255,0.050687,0.518156,0.557550,0.104152
5,merged_job_postings_di,nvidia_quarterly_revenue_clean,0.897423,0.889533,0.196728,6.455584,-0.203917,-0.006169,-0.031551,0.076278,-0.165358,0.002335,0.939582,0.699945,0.243776
